In [1]:
import os
import sys
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
DATA_RAW_DIR = "../data/raw"
DATA_PROCESSED_DIR = "../data/processed"

In [ ]:
df_train_full = pd.read_parquet(os.path.join(DATA_PROCESSED_DIR, 'train_featured.parquet'))
df_test_final = pd.read_parquet(os.path.join(DATA_PROCESSED_DIR, 'test_featured.parquet')) 

df_train, df_val = train_test_split(
    df_train_full, 
    test_size=0.20, 
    random_state=42, 
    stratify=df_train_full['is_fraud']
)

print(f"[1/3] Treino Puro (Passado): {df_train.shape}")
print(f"[2/3] Validação (Ajuste):   {df_val.shape}")
print(f"[3/3] Teste Final (Futuro):  {df_test_final.shape}")

[1/3] Treino Puro (Passado): (1037340, 29)
[2/3] Validação (Ajuste):   (259335, 29)
[3/3] Teste Final (Futuro):  (555719, 29)


In [3]:
from src.features.build_features import engineer_stateless_features

print("⏳ Construindo features comportamentais isoladas por partição...")
df_train_feat = engineer_stateless_features(df_train)
df_val_feat = engineer_stateless_features(df_val)
df_test_feat = engineer_stateless_features(df_test_final)

⏳ Construindo features comportamentais isoladas por partição...


In [8]:
from src.features.build_features import fit_apply_target_encoding

high_cardinality_cols = ['merchant', 'job', 'category']

print("Aplicando codificação de risco aprendida unicamente do Treino...")
train_ready, val_ready, test_ready = fit_apply_target_encoding(
    train_df=df_train_feat,
    val_df=df_val_feat,
    test_df=df_test_feat,
    columns=high_cardinality_cols,
    target='is_fraud'
)

Aplicando codificação de risco aprendida unicamente do Treino...


In [ ]:
features_for_model = [
    'amt', 'age', 'distance_to_merchant', 'trans_hour', 'trans_day_of_week',
    'time_since_last_trans', 'tx_count_24h', 'avg_amt_24h', 'amt_to_avg_ratio_24h',
    'merchant_encoded', 'job_encoded', 'category_encoded', 'is_fraud'
]

train_ready[features_for_model].to_parquet(os.path.join(DATA_PROCESSED_DIR, 'train_ready.parquet'), index=False)
val_ready[features_for_model].to_parquet(os.path.join(DATA_PROCESSED_DIR, 'val_ready.parquet'), index=False)
test_ready[features_for_model].to_parquet(os.path.join(DATA_PROCESSED_DIR, 'test_ready.parquet'), index=False)

print("Partições com sucesso em formato Parquet.")

Partições com sucesso em formato Parquet.
